In [1]:
# ============================================================
# RESNET50 SCRATCH (IMPROVED)
# ============================================================
import os, json, numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

from torchvision import transforms
from torchvision.models import resnet50

from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, roc_curve
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ============================================================
# PATHS
# ============================================================
BASE = "/kaggle/input/datasets/iharshsinha/deepfashion2-top5-processed/processed"

TRAIN_A = os.path.join(BASE,"train/annos")
TRAIN_I = os.path.join(BASE,"train/images")

VAL_A = os.path.join(BASE,"validation/annos")
VAL_I = os.path.join(BASE,"validation/images")

SAVE1="/kaggle/working/"
SAVE3="/kaggle/working/output/"
os.makedirs(SAVE3,exist_ok=True)

# ============================================================
# LABELS
# ============================================================
label_map = {
    "short sleeve top":0,
    "trousers":1,
    "shorts":2,
    "long sleeve top":3,
    "skirt":4
}
classes=list(label_map.keys())
num_classes=len(classes)

# ============================================================
# DATASET
# ============================================================
class DS(Dataset):
    def __init__(self,a_dir,i_dir,tf):
        self.files=[f for f in os.listdir(a_dir) if f.endswith(".json")]
        self.a=a_dir; self.i=i_dir; self.tf=tf

    def __len__(self): return len(self.files)

    def __getitem__(self,idx):
        f=self.files[idx]
        with open(os.path.join(self.a,f)) as x:
            data=json.load(x)

        y=np.zeros(num_classes)
        for k,v in data.items():
            if k.startswith("item"):
                c=v["category_name"]
                if c in label_map:
                    y[label_map[c]]=1

        img=Image.open(os.path.join(self.i,f.replace(".json",".jpg"))).convert("RGB")
        img=self.tf(img)

        return img, torch.tensor(y,dtype=torch.float32)

# ============================================================
# TRANSFORMS (IMPROVED)
# ============================================================
train_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.3,0.3),
    transforms.ToTensor()
])

val_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

# ============================================================
# LOADER
# ============================================================
train_loader = DataLoader(DS(TRAIN_A,TRAIN_I,train_tf),batch_size=64,shuffle=True,num_workers=2)
val_loader   = DataLoader(DS(VAL_A,VAL_I,val_tf),batch_size=64,num_workers=2)

# ============================================================
# MODEL (SCRATCH)
# ============================================================
model = resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features,num_classes)
model = model.to(device)

# ============================================================
# LOSS + OPT
# ============================================================
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=1e-4)
scaler = GradScaler()

# ============================================================
# TRAIN
# ============================================================
def train_epoch():
    model.train(); total=0
    for x,y in train_loader:
        x,y=x.to(device),y.to(device)
        optimizer.zero_grad()

        with autocast("cuda"):
            out=model(x)
            loss=criterion(out,y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total+=loss.item()

    return total/len(train_loader)

# ============================================================
# EVAL
# ============================================================
def eval_epoch():
    model.eval()
    P,T=[],[]
    with torch.no_grad():
        for x,y in val_loader:
            x=x.to(device)
            out=model(x)
            p=torch.sigmoid(out).cpu().numpy()
            P.append(p); T.append(y.numpy())

    P=np.vstack(P); T=np.vstack(T)
    Pb=(P>0.5).astype(int)

    print("\nPer Class Metrics")
    for i,c in enumerate(classes):
        print(c)
        print("P:",precision_score(T[:,i],Pb[:,i],zero_division=0))
        print("R:",recall_score(T[:,i],Pb[:,i],zero_division=0))
        print("F1:",f1_score(T[:,i],Pb[:,i],zero_division=0))
        print("AUC:",roc_auc_score(T[:,i],P[:,i]))
        print()

    return (
        f1_score(T,Pb,average="macro"),
        f1_score(T,Pb,average="micro"),
        precision_score(T,Pb,average="macro"),
        recall_score(T,Pb,average="macro"),
        roc_auc_score(T,P,average="macro"),
        P,T
    )

# ============================================================
# LOOP
# ============================================================
best=0
for e in range(5):
    loss=train_epoch()
    f1m,f1mi,p,r,auc,P,T = eval_epoch()

    print(f"\nEpoch {e+1}")
    print("Loss:",loss)
    print("Macro F1:",f1m)
    print("Micro F1:",f1mi)
    print("AUC:",auc)

    if f1m>best:
        best=f1m
        print("Saving Best Model")

        sd=model.state_dict()
        torch.save(sd,"resnet_scratch.pth")
        torch.save(sd,SAVE1+"resnet_scratch.pth")
        torch.save(sd,SAVE3+"resnet_scratch.pth")

Device: cuda

Per Class Metrics
short sleeve top
P: 0.6026474686483976
R: 0.8390590898068062
F1: 0.7014698428788647
AUC: 0.7124266351349267

trousers
P: 0.9169060320065654
R: 0.47071834843058774
F1: 0.6220768374164811
AUC: 0.863548461538268

shorts
P: 0.4030700241462573
R: 0.5654488265182676
F1: 0.4706474675259289
AUC: 0.7834307261423836

long sleeve top
P: 0.7476635514018691
R: 0.013522650439486139
F1: 0.026564834799933587
AUC: 0.7101260612577962

skirt
P: 0.675531914893617
R: 0.03908293583628251
F1: 0.07389090909090909
AUC: 0.7283705046196735


Epoch 1
Loss: 0.545581341120491
Macro F1: 0.3789299783424235
Micro F1: 0.5246137261947539
AUC: 0.7595804777386095
Saving Best Model

Per Class Metrics
short sleeve top
P: 0.8132421270005162
R: 0.509336351143804
F1: 0.6263730801729708
AUC: 0.7812702445437443

trousers
P: 0.7672169580931913
R: 0.8272593216768486
F1: 0.7961076478637676
AUC: 0.9164464590973284

shorts
P: 0.5523917995444191
R: 0.586740866198887
F1: 0.5690484571160389
AUC: 0.8509040

In [2]:
# ============================================================
# RESNET50 TRANSFER LEARNING (IMPROVED)
# ============================================================

model = resnet50(weights="IMAGENET1K_V1")

# freeze backbone
for p in model.parameters():
    p.requires_grad=False

model.fc = nn.Linear(model.fc.in_features,num_classes)
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(),lr=1e-3)
criterion = nn.BCEWithLogitsLoss()
scaler = GradScaler()

best=0

for e in range(5):
    loss=train_epoch()
    f1m,f1mi,p,r,auc,P,T = eval_epoch()

    print(f"\nEpoch {e+1}")
    print("Loss:",loss)
    print("Macro F1:",f1m)
    print("Micro F1:",f1mi)
    print("AUC:",auc)

    if f1m>best:
        best=f1m
        print("Saving Best Model")

        sd=model.state_dict()
        torch.save(sd,"resnet_transfer.pth")
        torch.save(sd,SAVE1+"resnet_transfer.pth")
        torch.save(sd,SAVE3+"resnet_transfer.pth")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 161MB/s]



Per Class Metrics
short sleeve top
P: 0.7897036249852403
R: 0.5406191900412255
F1: 0.6418426103646833
AUC: 0.775612884333072

trousers
P: 0.7780249660116179
R: 0.6630503475879503
F1: 0.7159510946829685
AUC: 0.8608940086492621

shorts
P: 0.5729537366548043
R: 0.38954754415678683
F1: 0.463776465504825
AUC: 0.8230573967893134

long sleeve top
P: 0.6338778409090909
R: 0.3017241379310345
F1: 0.4088410444342648
AUC: 0.7935408244644282

skirt
P: 0.7642477876106195
R: 0.33220495460840127
F1: 0.4631059631059631
AUC: 0.8428818401281977


Epoch 1
Loss: 0.49049594276119646
Macro F1: 0.538703435618541
Micro F1: 0.5845421291624622
AUC: 0.8191973908728546
Saving Best Model

Per Class Metrics
short sleeve top
P: 0.7363455379555094
R: 0.6876566162800097
F1: 0.7111687008861395
AUC: 0.7824231380067451

trousers
P: 0.8201497617426821
R: 0.6345060037918685
F1: 0.7154819169784429
AUC: 0.8714326399643096

shorts
P: 0.5357142857142857
R: 0.49721751754173726
F1: 0.5157485255364538
AUC: 0.8326040564431352

lon